In [0]:
# %pip install databricks-langchain==0.8.0 langchain==0.3.7 langchain-community==0.3.7 langchain-openai==0.2.6 databricks-labs-dqx==0.8.0

## Key Components in langchain_core.tools

The main tools and classes available:

* **BaseTool** - Base class for creating custom tools
* **Tool** - Simple tool wrapper for functions
* **StructuredTool** - Tool with structured input schema
* **@tool decorator** - Decorator to convert functions into tools
* **ToolException** - Exception class for tool errors
* **create_retriever_tool** - Create tools from retrievers
* **convert_runnable_to_tool** - Convert runnables to tools
* **BaseToolkit** - Base class for tool collections

In [0]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

print(f"Tool name: {multiply.name}")
print(f"Tool description: {multiply.description}")
print(f"Tool result: {multiply.invoke({'a': 5, 'b': 3})}")

In [0]:
from langchain_core.tools import Tool

def search_function(query: str) -> str:
    """Simulates a search function."""
    return f"Search results for: {query}"

search_tool = Tool(
    name="search",
    description="Useful for searching information",
    func=search_function
)

print(f"Tool name: {search_tool.name}")
print(f"Result: {search_tool.invoke('langchain')}")

In [0]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class CalculatorInput(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")
    operation: str = Field(description="Operation: add, subtract, multiply, divide")

def calculator(a: int, b: int, operation: str) -> float:
    """Perform basic arithmetic operations."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b if b != 0 else "Error: Division by zero"
    return "Unknown operation"

calc_tool = StructuredTool.from_function(
    func=calculator,
    name="calculator",
    description="Performs basic arithmetic operations",
    args_schema=CalculatorInput
)

print(f"Tool name: {calc_tool.name}")
print(f"Input schema: {calc_tool.args_schema.schema()}")
print(f"Result: {calc_tool.invoke({'a': 10, 'b': 5, 'operation': 'multiply'})}")

In [0]:
from langchain_core.tools import BaseTool
from typing import Optional, Type
from pydantic import BaseModel, Field

class TextAnalysisInput(BaseModel):
    text: str = Field(description="Text to analyze")

class TextAnalysisTool(BaseTool):
    name: str = "text_analyzer"
    description: str = "Analyzes text and returns word count, character count, and sentence count"
    args_schema: Type[BaseModel] = TextAnalysisInput
    
    def _run(self, text: str) -> dict:
        """Synchronous implementation."""
        words = len(text.split())
        chars = len(text)
        sentences = text.count('.') + text.count('!') + text.count('?')
        return {
            "word_count": words,
            "char_count": chars,
            "sentence_count": sentences
        }
    
    async def _arun(self, text: str) -> dict:
        """Async implementation (optional)."""
        return self._run(text)

analyzer = TextAnalysisTool()
result = analyzer.invoke({"text": "Hello world! This is a test. How are you?"})
print(f"Analysis result: {result}")

## LangChain Community Tools

Popular tools available in `langchain_community.tools`:

* **DuckDuckGoSearchRun** - Web search using DuckDuckGo
* **WikipediaQueryRun** - Search Wikipedia articles
* **ArxivQueryRun** - Search academic papers on ArXiv
* **PythonREPLTool** - Execute Python code
* **ShellTool** - Execute shell commands
* **RequestsGetTool** - Make HTTP GET requests
* **HumanInputRun** - Get input from humans
* **FileManagementToolkit** - File operations (read, write, list)

Note: Some tools require additional packages to be installed.

In [0]:
%pip install ddgs wikipedia arxiv --quiet

In [0]:
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun

# Create the search tool
search = DuckDuckGoSearchRun()

print("Tool name:", search.name)
print("Tool description:", search.description)
print("\n" + "="*50)
print("Search Results:")
print("="*50)

# Perform a search
result = search.invoke("Artificial Intelligence")
print(result)

In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Create Wikipedia tool with custom configuration
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=1,
        doc_content_chars_max=500
    )
)

print("Tool name:", wikipedia.name)
print("\n" + "="*50)
print("Wikipedia Results:")
print("="*50)

# Search Wikipedia
result = wikipedia.invoke("Artificial Intelligence")
print(result)

In [0]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

# Create ArXiv tool
arxiv = ArxivQueryRun(
    api_wrapper=ArxivAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=500
    )
)

print("Tool name:", arxiv.name)
print("\n" + "="*50)
print("ArXiv Results:")
print("="*50)

# Search for papers
result = arxiv.invoke("large language models")
print(result)

In [0]:
from langchain_community.tools.python_repl import PythonREPLTool

# Create Python REPL tool
python_repl = PythonREPLTool()

print("Tool name:", python_repl.name)
print("Tool description:", python_repl.description)
print("\n" + "="*50)
print("Execution Results:")
print("="*50)

# Execute Python code
code = """
import math
result = math.sqrt(144) + math.pi
print(f"Square root of 144 plus pi: {result:.2f}")
"""

result = python_repl.invoke(code)
print(result)

In [0]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Create a list of tools
tools = [
    DuckDuckGoSearchRun(),
    WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)),
]

print("Available Tools:")
print("="*50)
for tool in tools:
    print(f"\n• {tool.name}")
    print(f"  Description: {tool.description}")
    print(f"  Type: {type(tool).__name__}")

print("\n" + "="*50)
print("Example: Using tools to research a topic")
print("="*50)

topic = "Databricks"
print(f"\nSearching for: {topic}\n")

for tool in tools:
    print(f"\n--- Using {tool.name} ---")
    try:
        result = tool.invoke(topic)
        print(result[:300] + "..." if len(result) > 300 else result)
    except Exception as e:
        print(f"Error: {e}")

In [0]:
from langchain_community.tools import DuckDuckGoSearchRun
import json

search_tool = DuckDuckGoSearchRun()

print("Tool Metadata:")
print("="*50)
print(f"Name: {search_tool.name}")
print(f"Description: {search_tool.description}")
print(f"Return Direct: {search_tool.return_direct}")
print(f"\nArgs Schema:")
if hasattr(search_tool, 'args_schema') and search_tool.args_schema:
    print(json.dumps(search_tool.args_schema.model_json_schema(), indent=2))
else:
    print("No structured schema defined")

print(f"\nTool can be used with LangChain agents to:")
print("• Provide real-time information")
print("• Answer questions requiring current data")
print("• Augment LLM knowledge with external sources")